# AdventureWorks Star Schema Load

Promotes the bronze tables in `adventure_works.bronze` (loaded from the raw headerless TSVs by the Bronze Ingestion notebook) to the fact/dim star schema in `adventure_works.silver_v2` (fresh lineage identity, rebuilt purely from bronze), selecting the columns in the order defined by the `CREATE TABLE` statements in `instawdb.sql`.

Notes:
* Silver is built **from the bronze layer**, not from the raw volume (raw files are tab-separated with no header row).
* Unity Catalog stores table names in lowercase, so `dim_PurchaseOrderDetail` is saved as `dim_purchaseorderdetail`.

In [0]:
# Configuration — silver is built from the bronze layer
bronze_schema = "adventure_works.bronze"
target_schema = "adventure_works.silver_v2"

# Bronze table -> (silver table, column names in CREATE TABLE order from instawdb.sql)
table_configs = {
    "fact_sales": (
        "fact_sales",
        ["SalesOrderID", "SalesOrderDetailID", "CarrierTrackingNumber", "OrderQty",
         "ProductID", "SpecialOfferID", "UnitPrice", "UnitPriceDiscount", "LineTotal",
         "rowguid", "ModifiedDate"],
    ),
    "dim_product": (
        "dim_product",
        ["ProductID", "Name", "ProductNumber", "MakeFlag", "FinishedGoodsFlag", "Color",
         "SafetyStockLevel", "ReorderPoint", "StandardCost", "ListPrice", "Size",
         "SizeUnitMeasureCode", "WeightUnitMeasureCode", "Weight", "DaysToManufacture",
         "ProductLine", "Class", "Style", "ProductSubcategoryID", "ProductModelID",
         "SellStartDate", "SellEndDate", "DiscontinuedDate", "rowguid", "ModifiedDate"],
    ),
    "dim_sales_territory": (
        "dim_sales_territory",
        ["TerritoryID", "Name", "CountryRegionCode", "Group", "SalesYTD", "SalesLastYear",
         "CostYTD", "CostLastYear", "rowguid", "ModifiedDate"],
    ),
    "dim_purchaseorderdetail": (
        "dim_purchaseorderdetail",
        ["PurchaseOrderID", "PurchaseOrderDetailID", "DueDate", "OrderQty", "ProductID",
         "UnitPrice", "LineTotal", "ReceivedQty", "RejectedQty", "StockedQty", "ModifiedDate"],
    ),
    "dim_purchaseorderheader": (
        "dim_purchaseorderheader",
        ["PurchaseOrderID", "RevisionNumber", "Status", "EmployeeID", "VendorID",
         "ShipMethodID", "OrderDate", "ShipDate", "SubTotal", "TaxAmt", "Freight",
         "TotalDue", "ModifiedDate"],
    ),
    "dim_vendor": (
        "dim_vendor",
        ["BusinessEntityID", "AccountNumber", "Name", "CreditRating", "PreferredVendorStatus",
         "ActiveFlag", "PurchasingWebServiceURL", "ModifiedDate"],
    ),
}

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

for bronze_table, (silver_table, columns) in table_configs.items():
    # Silver is built from the bronze layer (not the raw volume): select the
    # bronze columns in the CREATE TABLE order from instawdb.sql and promote
    # them to the silver fact/dim tables.
    df = spark.table(f"{bronze_schema}.{bronze_table}").select(*columns)

    rows_written = df.count()

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{target_schema}.{silver_table}"))

    print(f"{silver_table}: {rows_written:,} rows written from {bronze_schema}.{bronze_table}")

fact_sales: 121,317 rows written from adventure_works.bronze.fact_sales
dim_product: 504 rows written from adventure_works.bronze.dim_product
dim_sales_territory: 10 rows written from adventure_works.bronze.dim_sales_territory
dim_purchaseorderdetail: 8,845 rows written from adventure_works.bronze.dim_purchaseorderdetail
dim_purchaseorderheader: 4,012 rows written from adventure_works.bronze.dim_purchaseorderheader
dim_vendor: 104 rows written from adventure_works.bronze.dim_vendor


In [0]:
# Expected rows = bronze source counts, to confirm nothing was lost or duplicated
expected_rows = {
    "fact_sales": 121317,
    "dim_product": 504,
    "dim_sales_territory": 10,
    "dim_purchaseorderdetail": 8845,
    "dim_purchaseorderheader": 4012,
    "dim_vendor": 104,
}

verification_rows = []
for bronze_table, (silver_table, columns) in table_configs.items():
    loaded = spark.table(f"{target_schema}.{silver_table}")
    verification_rows.append((
        silver_table,
        expected_rows[silver_table],
        loaded.count(),
        loaded.columns == columns,
    ))

verification_df = spark.createDataFrame(
    verification_rows,
    schema="table string, expected_rows long, loaded_rows long, columns_match boolean",
)
display(verification_df)

bad = verification_df.filter("loaded_rows != expected_rows OR NOT columns_match")
assert bad.count() == 0, f"Mismatches found: {bad.collect()}"
print("Success: all 6 silver fact/dim tables loaded from bronze with the correct column names.")

table,expected_rows,loaded_rows,columns_match
fact_sales,121317,121317,true
dim_product,504,504,true
dim_sales_territory,10,10,true
dim_purchaseorderdetail,8845,8845,true
dim_purchaseorderheader,4012,4012,true
dim_vendor,104,104,true


Success: all 6 silver fact/dim tables loaded from bronze with the correct column names.
